In [ ]:
# Install webdriver_manager if not already installed
!pip install webdriver-manager selenium

import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager

# --- Installation and setup for Google Chrome and ChromeDriver ---
# Clean up old chromium-browser and chromedriver to avoid conflicts
!apt-get update -qq > /dev/null
!apt-get purge -y chromium-browser chromium-chromedriver > /dev/null 2>&1
!rm -rf /usr/bin/chromium-browser /usr/bin/chromedriver > /dev/null 2>&1

# Add Google Chrome repository and install Google Chrome stable
# This ensures we get a recent and stable version of Chrome
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" | tee /etc/apt/sources.list.d/google-chrome.list
!apt-get update -qq > /dev/null
!apt-get install -y google-chrome-stable > /dev/null

# Cấu hình bắt buộc cho Colab
options = Options()
options.add_argument('--headless=new') # Use 'new' headless mode for better performance
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--window-size=1920,1080') # Recommended for headless browsers
options.binary_location = '/usr/bin/google-chrome' # Specify path to Google Chrome Stable

# Đường dẫn mặc định của chromedriver trên Linux (Colab)
# Use ChromeDriverManager to automatically download and manage chromedriver
service = ChromeService(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)

driver.get("https://www.google.com")
print("Đã chạy thành công trên Colab. Tiêu đề:", driver.title)
driver.quit()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 5.8 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
OK
deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
W: http://dl.google.com/linux/chrome/deb/dists/stable/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION sect

In [ ]:
!apt update
!apt install -y chromium-browser chromedriver

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:10 https://dl.google.com/linux/chrome/deb stable InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
118 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry m

In [ ]:
#import
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException
from webdriver_manager.chrome import ChromeDriverManager

import pandas as pd
import time
import re
from google.colab import files

options = Options()
options.add_argument('--headless=new')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--window-size=1920,1080')
options.add_argument(
    'user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
    'AppleWebKit/537.36 (KHTML, like Gecko) '
    'Chrome/122.0.0.0 Safari/537.36'
)
options.binary_location = '/usr/bin/google-chrome'

service = ChromeService(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)

# --- PHẦN ĐỌC DỮ LIỆU TỪ FILE CSV ---
# Thay tên file bạn muốn chạy vào đây (ví dụ: file '2024.csv')
input_file = "/content/Danh sách khách sạn 1-5 sao.xlsx"

# Nếu lỗi, thử đọc bằng utf-16 (thường gặp khi xuất từ Excel)
df_input = pd.read_excel(input_file)

# Tạo hotel_list từ file
hotel_list = []
for _, row in df_input.iterrows():
    hotel_list.append({
        "name": str(row['Tên khách sạn']).strip(),
        "url": str(row['link']).strip()
    })

print(f"✅ Đã nạp thành công {len(hotel_list)} khách sạn từ file.")
# ------------------------------------

def scroll_to_bottom():
    last_height = driver.execute_script("return document.body.scrollHeight")

    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3)

        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break

        last_height = new_height

# next
def click_all_next_pages():
    print("🔄 Bắt đầu bấm Next cho đến khi hết...")
    page_count = 0

    while True:
        try:
            next_btn = driver.find_element( By.XPATH, '//button[@class="Review-paginator-button"]' )

            driver.execute_script("arguments[0].click();", next_btn)
            time.sleep(5)

            scroll_to_bottom()
            time.sleep(10)

            page_count += 1
            print(f"➡️  Đã bấm Next {page_count} lần")
            if page_count >= 30:
                break

        except NoSuchElementException:
            print("✅ Đã hết nút Next")
            break

        except Exception as e:
            print("⚠️ Lỗi khi bấm Next:", e)
            break

#lấy review
def get_reviews_from_url(hotel_name, url):
    print(f"\n🏨 Đang xử lý: {hotel_name}")
    driver.get(url)
    time.sleep(8)

    scroll_to_bottom()
    time.sleep(4)

    click_all_next_pages()
    time.sleep(10)

    print("📥 Bắt đầu thu thập review...")
    hotel_reviews = []
    seen_content = set()

    time.sleep(10)
    cards = driver.find_elements(By.CLASS_NAME, "Review-comment")
    print("Tổng số review card tìm thấy:", len(cards))

    for card in cards:
        try:
            content_node = card.find_element(By.CLASS_NAME, "Review-comment-right")
            full_text = content_node.text

            clean_text = re.split(
                r'Phản hồi từ khách sạn|Response from the hotel',
                full_text,
                flags=re.IGNORECASE
            )[0].strip().replace('\n', ' ')

            # if clean_text not in seen_content and is_vietnamese(clean_text):
            if clean_text not in seen_content:
                rating = card.find_element(
                    By.CLASS_NAME,
                    "Review-comment-leftHeader"
                ).text
                hotel_reviews.append({
                    "Hotel": hotel_name,
                    "Rating": rating,
                    "Content": clean_text
                })
                seen_content.add(clean_text)
        except:
            continue
    print(f"🎯 Thu được {len(hotel_reviews)} review")
    return hotel_reviews


#chạy tổng hợp
final_data = []
for hotel in hotel_list:
    reviews = get_reviews_from_url(hotel['name'], hotel['url'])
    final_data.extend(reviews)

#lưu file
if final_data:
    df = pd.DataFrame(final_data)
    df.to_csv('tat_ca_review_khach_san.csv', index=False, encoding='utf-16')

    print(f"\n✅ HOÀN THÀNH! Tổng cộng thu được {len(df)} đánh giá.")
else:
    print("❌ Không thu được review nào.")
driver.quit()

✅ Đã nạp thành công 100 khách sạn từ file.

🏨 Đang xử lý: KHÁCH SẠN THE MYST ĐỒNG KHỞI
🔄 Bắt đầu bấm Next cho đến khi hết...
➡️  Đã bấm Next 1 lần
➡️  Đã bấm Next 2 lần
➡️  Đã bấm Next 3 lần
➡️  Đã bấm Next 4 lần
➡️  Đã bấm Next 5 lần
➡️  Đã bấm Next 6 lần
➡️  Đã bấm Next 7 lần
➡️  Đã bấm Next 8 lần
➡️  Đã bấm Next 9 lần
➡️  Đã bấm Next 10 lần
➡️  Đã bấm Next 11 lần
➡️  Đã bấm Next 12 lần
➡️  Đã bấm Next 13 lần
➡️  Đã bấm Next 14 lần
➡️  Đã bấm Next 15 lần
➡️  Đã bấm Next 16 lần
➡️  Đã bấm Next 17 lần
➡️  Đã bấm Next 18 lần
➡️  Đã bấm Next 19 lần
➡️  Đã bấm Next 20 lần
➡️  Đã bấm Next 21 lần
➡️  Đã bấm Next 22 lần
➡️  Đã bấm Next 23 lần
➡️  Đã bấm Next 24 lần
✅ Đã hết nút Next
📥 Bắt đầu thu thập review...
Tổng số review card tìm thấy: 1719
🎯 Thu được 1716 review

🏨 Đang xử lý: KHÁCH SẠN MAI HOUSE SAIGON
🔄 Bắt đầu bấm Next cho đến khi hết...
➡️  Đã bấm Next 1 lần
➡️  Đã bấm Next 2 lần
➡️  Đã bấm Next 3 lần
➡️  Đã bấm Next 4 lần
➡️  Đã bấm Next 5 lần
➡️  Đã bấm Next 6 lần
➡️  Đã bấm Next

In [ ]:
import pandas as pd

# Đọc file CSV
df = pd.read_csv('tat_ca_review_khach_san.csv', encoding='utf-16')

# Xem 5 dòng đầu tiên
print(df.head())

# Kiểm tra tổng số dòng và cột
print("\nKích thước file:", df.shape)

                          Hotel                  Rating  \
0  KHÁCH SẠN THE MYST ĐỒNG KHỞI  10,0 Trên cả tuyệt vời   
1  KHÁCH SẠN THE MYST ĐỒNG KHỞI  10,0 Trên cả tuyệt vời   
2  KHÁCH SẠN THE MYST ĐỒNG KHỞI   9,6 Trên cả tuyệt vời   
3  KHÁCH SẠN THE MYST ĐỒNG KHỞI   9,2 Trên cả tuyệt vời   
4  KHÁCH SẠN THE MYST ĐỒNG KHỞI             7,6 Rất tốt   

                                             Content  
0  “Trên cả tuyệt vời” Cách phố đi bộ Nguyễn Huệ ...  
1  “Khách sạn tuyệt vời” Tôi đã có chuyến đi tuần...  
2  “Căn cứ tuyệt vời tại quận 1, TP. HCM” Một khá...  
3  “Cảm thấy thật được chiều chuộng ở đây!” Chúng...  
4  “Hãy chắc chắn rằng bạn nhận được những gì bạn...  

Kích thước file: (77783, 3)


In [ ]:
from google.colab import files

# Lệnh này sẽ kích hoạt trình duyệt tự động tải file về máy bạn
files.download('tat_ca_review_khach_san.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>